# The settings that decide whether it trains at all

MichAl Academy, unit 3.5.

Run each cell with **Shift+Enter**.

Lesson 3.2 produced the gradients. Nothing so far says how big a step to take,
how many examples to look at first, or what exactly to measure. This notebook
runs all three questions on one network so the answers can be compared.


In [ ]:
import time

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.set_num_threads(1)

X, y = load_digits(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

scaler = StandardScaler().fit(Xtr)
Xtr_t = torch.tensor(scaler.transform(Xtr), dtype=torch.float32)
Xte_t = torch.tensor(scaler.transform(Xte), dtype=torch.float32)
ytr_t, yte_t = torch.tensor(ytr), torch.tensor(yte)
ONEHOT = torch.nn.functional.one_hot(ytr_t, 10).float()

print(f"digits: {X.shape}, train {len(Xtr)}, test {len(Xte)}")


In [ ]:
def make_net(seed):
    torch.manual_seed(seed)
    return torch.nn.Sequential(
        torch.nn.Linear(64, 32), torch.nn.ReLU(), torch.nn.Linear(32, 10))


def run(opt_name, lr, seed=0, epochs=40, batch=64, loss_name="ce"):
    """Train once and report test accuracy plus the training-accuracy curve."""
    net = make_net(seed)
    if opt_name == "sgd":
        opt = torch.optim.SGD(net.parameters(), lr=lr)
    elif opt_name == "momentum":
        opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    else:
        opt = torch.optim.Adam(net.parameters(), lr=lr)

    ce = torch.nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(seed)
    curve = []
    for _ in range(epochs):
        perm = torch.randperm(len(Xtr_t), generator=g)
        for i in range(0, len(Xtr_t), batch):
            idx = perm[i:i + batch]
            opt.zero_grad()
            out = net(Xtr_t[idx])
            if loss_name == "ce":
                loss = ce(out, ytr_t[idx])
            elif loss_name == "mse_softmax":
                loss = ((torch.softmax(out, 1) - ONEHOT[idx]) ** 2).mean()
            else:
                loss = ((torch.sigmoid(out) - ONEHOT[idx]) ** 2).mean()
            loss.backward()
            opt.step()
        with torch.no_grad():
            curve.append((net(Xtr_t).argmax(1) == ytr_t).float().mean().item())
    with torch.no_grad():
        test = (net(Xte_t).argmax(1) == yte_t).float().mean().item()
    return test, curve


## 1. The learning rate cliff

Same network, same data, three optimisers, six learning rates spanning five
orders of magnitude. Ten percent is exactly chance on ten balanced classes, so
watch for it.

This cell trains 54 networks and is the slow one.


In [ ]:
SEEDS = 3
LRS = (1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0)
OPTS = ("sgd", "momentum", "adam")

grid = {}
print(f"{'lr':<10}" + "".join(f"{o:>12}" for o in OPTS))
for lr in LRS:
    row = []
    for opt_name in OPTS:
        med = float(np.median([run(opt_name, lr, s)[0] for s in range(SEEDS)]))
        grid[(opt_name, lr)] = med
        row.append(med)
    print(f"{lr:<10g}" + "".join(f"{v:>12.4f}" for v in row))


In [ ]:
for opt_name in OPTS:
    scores = [(grid[(opt_name, lr)], lr) for lr in LRS]
    best_acc, best_lr = max(scores)
    worst_acc, worst_lr = min(scores)
    print(f"{opt_name:<9} best {best_acc:.4f} at lr={best_lr:<8g}"
          f"   worst anywhere {worst_acc:.4f} at lr={worst_lr:g}")


Three things worth more than the headline.

**At its own best setting each optimiser is the same.** The spread between the
three best scores is under half a point. Adam is not buying accuracy.

**Momentum's cliff is closer than plain SGD's.** At a learning rate of 1 plain
SGD is still working and momentum has collapsed to chance. Momentum keeps a
running average of past gradients, so the step it takes is larger than the one
you asked for. Speed costs headroom.

**Adam never reaches chance**, even at a learning rate a hundred times its best.
That is the real difference: not a better answer, a much wider range of settings
that still produces one.

## 2. Batch size is a claim about steps, not epochs

An epoch is one pass over the data. It sounds like a unit of work. It is not a
unit of learning.


In [ ]:
print(f"{'batch':<10}{'steps/epoch':<14}{'time/epoch':<14}{'test acc':<12}epochs to .99 train")
for batch in (8, 32, 128, len(Xtr)):
    start = time.time()
    test, curve = run("adam", 1e-3, 0, epochs=40, batch=batch)
    per_epoch = (time.time() - start) / 40
    reached = next((i + 1 for i, a in enumerate(curve) if a >= 0.99), None)
    steps = int(np.ceil(len(Xtr) / batch))
    print(f"{('full' if batch == len(Xtr) else batch):<10}{steps:<14}"
          f"{per_epoch * 1e3:<14.1f}{test:<12.4f}{reached if reached else 'not reached'}")


In [ ]:
for batch in (8, len(Xtr)):
    steps = int(np.ceil(len(Xtr) / batch)) * 40
    print(f"batch {('full' if batch == len(Xtr) else batch):>5}: "
          f"{steps} weight updates in 40 epochs")


Full batch is much cheaper per epoch and ends up far worse, because forty epochs
of full batch is forty weight updates. Its gradient is the most accurate of the
four, which is exactly the point: an accurate direction you only follow forty
times gets you nowhere.

## 3. The loss function rule, and when it is true

You will be told to use cross-entropy for classification and never squared
error. The reason usually given is that squared error stops pushing when the
model is confidently wrong. Put a number on that first.


In [ ]:
# One output that should be 1, sitting deep in the flat region of a sigmoid.
z = torch.tensor([-6.0], requires_grad=True)
((torch.sigmoid(z) - 1.0) ** 2).backward()
mse_grad = z.grad.item()

z2 = torch.tensor([-6.0], requires_grad=True)
torch.nn.functional.binary_cross_entropy_with_logits(z2, torch.tensor([1.0])).backward()
ce_grad = z2.grad.item()

print(f"output is {torch.sigmoid(torch.tensor(-6.0)).item():.4f}, target is 1.0")
print(f"  squared error gradient : {mse_grad:.6f}")
print(f"  cross-entropy gradient : {ce_grad:.6f}")
print(f"  cross-entropy pushes {abs(ce_grad / mse_grad):.0f}x harder")


That is the argument, and it is real. Now the part that gets left out: it only
applies if the outputs actually saturate.


In [ ]:
SEEDS_LOSS = 15
print(f"{'loss':<14}{'median test':<16}{'median epochs to .99':<24}never reached")
for loss_name in ("ce", "mse_softmax", "mse_sigmoid"):
    tests, reached, never = [], [], 0
    for s in range(SEEDS_LOSS):
        test, curve = run("adam", 1e-3, s, epochs=40, loss_name=loss_name)
        tests.append(test)
        hit = next((i + 1 for i, a in enumerate(curve) if a >= 0.99), None)
        if hit is None:
            never += 1
        else:
            reached.append(hit)
    print(f"{loss_name:<14}{np.median(tests):<16.4f}"
          f"{(f'{np.median(reached):.0f}' if reached else 'n/a'):<24}{never} of {SEEDS_LOSS}")


In [ ]:
# Are this network's outputs anywhere near saturated to begin with?
torch.manual_seed(0)
probe = make_net(0)
with torch.no_grad():
    logits = probe(Xtr_t)
print(f"at initialisation, largest |output| is {logits.abs().max().item():.2f}")
print("nowhere near the flat region, which is why squared error on a softmax was fine")


Squared error on a softmax was not worse here. It was slightly faster. The
saturation the argument depends on never happened.

Put the same squared error on a sigmoid output, where units do saturate, and it
never reached 99% training accuracy in any of the fifteen runs.

The rule survives and is now one you can reason about instead of recite. Use
cross-entropy, because on the day your outputs do saturate, squared error stops
learning from its worst mistakes.

## 4. What you have

- Every optimiser has a learning rate cliff. Changing optimiser moves the cliff,
  it does not remove it, and momentum moves it closer.
- Adam's value is tolerance, not accuracy.
- Batch size decides how many times the weights actually move. Epochs are not
  the unit that matters.
- Cross-entropy is the right default for a reason that also tells you when it
  would not matter.

Lesson 3.4 changes the wiring for data that has a shape.
